In [1]:
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, regexp_extract

In [2]:
# 1) START SPARK SESSION

spark = (
    SparkSession.builder
    .appName("DiabetesReadmissionFullData")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

26/04/16 20:34:50 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
# 2) FILE PATH

FULL_DATA_PATH = "gs://cs777-bucket-derek/diabetic_data.csv"

In [4]:
# 3) LOAD FULL DATASET FROM GCS

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(FULL_DATA_PATH)
)

print("Full dataset row count:", df.count())
print("Full dataset columns:", len(df.columns))
df.printSchema()

Full dataset row count: 101766


Full dataset columns: 50
root
 |-- encounter_id: integer (nullable = true)
 |-- patient_nbr: integer (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: string (nullable = true)
 |-- weight: string (nullable = true)
 |-- admission_type_id: integer (nullable = true)
 |-- discharge_disposition_id: integer (nullable = true)
 |-- admission_source_id: integer (nullable = true)
 |-- time_in_hospital: integer (nullable = true)
 |-- payer_code: string (nullable = true)
 |-- medical_specialty: string (nullable = true)
 |-- num_lab_procedures: integer (nullable = true)
 |-- num_procedures: integer (nullable = true)
 |-- num_medications: integer (nullable = true)
 |-- number_outpatient: integer (nullable = true)
 |-- number_emergency: integer (nullable = true)
 |-- number_inpatient: integer (nullable = true)
 |-- diag_1: string (nullable = true)
 |-- diag_2: string (nullable = true)
 |-- diag_3: string (nullable = true)
 |-- number_diagnoses: inte

In [5]:
# 4) CREATE DIAGNOSIS GROUP FEATURE FROM diag_1

df = df.withColumn(
    "diag_1_str",
    col("diag_1").cast("string")
)

df = df.withColumn(
    "diag_1_num_str",
    regexp_extract(col("diag_1_str"), r"^([0-9]+)", 1)
)

df = df.withColumn(
    "diag_1_num",
    when(col("diag_1_num_str") != "", col("diag_1_num_str").cast("int"))
)

df = df.withColumn(
    "diag_1_group",
    when(col("diag_1_str").startswith("250"), 1.0)   # diabetes
    .when(col("diag_1_num").between(390, 459), 2.0)  # circulatory
    .when(col("diag_1_num") == 785, 2.0)             # circulatory-related symptoms
    .when(col("diag_1_num").between(460, 519), 3.0)  # respiratory
    .when(col("diag_1_num") == 786, 3.0)             # respiratory-related symptoms
    .when(col("diag_1_num").between(520, 579), 4.0)  # digestive
    .when(col("diag_1_num") == 787, 4.0)             # digestive-related symptoms
    .when(col("diag_1_num").between(580, 629), 5.0)  # genitourinary
    .when(col("diag_1_num") == 788, 5.0)             # genitourinary-related symptoms
    .when(col("diag_1_num").between(800, 999), 6.0)  # injury / poisoning
    .when(col("diag_1_num").between(710, 739), 7.0)  # musculoskeletal
    .when(col("diag_1_num").between(140, 239), 8.0)  # neoplasms
    .otherwise(0.0)                                  # other / unknown
)

print("Diagnosis group counts:")
df.groupBy("diag_1_group").count().orderBy("diag_1_group").show()

Diagnosis group counts:


+------------+-----+
|diag_1_group|count|
+------------+-----+
|         0.0|18193|
|         1.0| 8757|
|         2.0|30437|
|         3.0|14423|
|         4.0| 9475|
|         5.0| 5117|
|         6.0| 6974|
|         7.0| 4957|
|         8.0| 3433|
+------------+-----+



In [6]:
# 5) SELECT FEATURES

selected_cols = [
    # Core utilization
    "time_in_hospital",
    "num_medications",
    "number_inpatient",
    "number_emergency",
    "number_outpatient",

    # Additional numeric severity features
    "num_lab_procedures",
    "number_diagnoses",

    # Clinical
    "A1Cresult",
    "max_glu_serum",
    "diag_1_group",

    # Demographics
    "age",
    "gender",

    # Admission context
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",

    # Diabetes treatment
    "diabetesMed",
    "change",
    "insulin",

    # Target
    "readmitted"
]

df = df.select(*selected_cols)

In [7]:
# 6) CREATE BINARY LABEL
# <30 = 1
# NO and >30 = 0

df = df.withColumn(
    "label",
    when(col("readmitted") == "<30", 1.0).otherwise(0.0)
)

In [8]:
# 7) CLEAN MISSING VALUES

df = df.replace("?", None)

df = df.fillna({
    "A1Cresult": "None",
    "max_glu_serum": "None",
    "age": "[50-60)",
    "gender": "Unknown",
    "diabetesMed": "No",
    "change": "No",
    "insulin": "No"
})

numeric_required = [
    "time_in_hospital",
    "num_medications",
    "number_inpatient",
    "number_emergency",
    "number_outpatient",
    "num_lab_procedures",
    "number_diagnoses",
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

df = df.dropna(subset=numeric_required)

print("Row count after cleaning:", df.count())

Row count after cleaning: 101766


In [9]:
# 8) MANUAL ENCODING OF CATEGORICAL FEATURES

# age
df = df.withColumn(
    "age_num",
    when(col("age") == "[0-10)", 0.0)
    .when(col("age") == "[10-20)", 1.0)
    .when(col("age") == "[20-30)", 2.0)
    .when(col("age") == "[30-40)", 3.0)
    .when(col("age") == "[40-50)", 4.0)
    .when(col("age") == "[50-60)", 5.0)
    .when(col("age") == "[60-70)", 6.0)
    .when(col("age") == "[70-80)", 7.0)
    .when(col("age") == "[80-90)", 8.0)
    .when(col("age") == "[90-100)", 9.0)
    .otherwise(5.0)
)

# gender
df = df.withColumn(
    "gender_num",
    when(col("gender") == "Male", 1.0)
    .when(col("gender") == "Female", 0.0)
    .otherwise(0.5)
)

# A1Cresult
df = df.withColumn(
    "A1Cresult_num",
    when(col("A1Cresult") == "None", 0.0)
    .when(col("A1Cresult") == "Norm", 1.0)
    .when(col("A1Cresult") == ">7", 2.0)
    .when(col("A1Cresult") == ">8", 3.0)
    .otherwise(0.0)
)

# max_glu_serum
df = df.withColumn(
    "max_glu_serum_num",
    when(col("max_glu_serum") == "None", 0.0)
    .when(col("max_glu_serum") == "Norm", 1.0)
    .when(col("max_glu_serum") == ">200", 2.0)
    .when(col("max_glu_serum") == ">300", 3.0)
    .otherwise(0.0)
)

# diabetesMed
df = df.withColumn(
    "diabetesMed_num",
    when(col("diabetesMed") == "Yes", 1.0).otherwise(0.0)
)

# change
df = df.withColumn(
    "change_num",
    when(col("change") == "Ch", 1.0).otherwise(0.0)
)

# insulin
df = df.withColumn(
    "insulin_num",
    when(col("insulin") == "No", 0.0)
    .when(col("insulin") == "Steady", 1.0)
    .when(col("insulin") == "Up", 2.0)
    .when(col("insulin") == "Down", 3.0)
    .otherwise(0.0)
)

In [10]:
# 9) KEEP FINAL MODEL COLUMNS

model_cols = [
    "time_in_hospital",
    "num_medications",
    "number_inpatient",
    "number_emergency",
    "number_outpatient",
    "num_lab_procedures",
    "number_diagnoses",
    "A1Cresult_num",
    "max_glu_serum_num",
    "diag_1_group",
    "age_num",
    "gender_num",
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "diabetesMed_num",
    "change_num",
    "insulin_num",
    "label"
]

df_model = df.select(*model_cols)

print("Final modeling row count:", df_model.count())
df_model.show(5, truncate=False)

Final modeling row count: 101766


+----------------+---------------+----------------+----------------+-----------------+------------------+----------------+-------------+-----------------+------------+-------+----------+-----------------+------------------------+-------------------+---------------+----------+-----------+-----+
|time_in_hospital|num_medications|number_inpatient|number_emergency|number_outpatient|num_lab_procedures|number_diagnoses|A1Cresult_num|max_glu_serum_num|diag_1_group|age_num|gender_num|admission_type_id|discharge_disposition_id|admission_source_id|diabetesMed_num|change_num|insulin_num|label|
+----------------+---------------+----------------+----------------+-----------------+------------------+----------------+-------------+-----------------+------------+-------+----------+-----------------+------------------------+-------------------+---------------+----------+-----------+-----+
|1               |1              |0               |0               |0                |41                |1         

In [11]:
# 10) TRAIN / TEST SPLIT

train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=42)

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())

Train rows: 81391


Test rows: 20375


In [12]:
# 11) CONVERT DATAFRAME TO RDD OF (x, y)

feature_names = model_cols[:-1]
num_features = len(feature_names)

def row_to_xy(row):
    x = np.array([float(row[c]) for c in feature_names], dtype=float)
    y = float(row["label"])
    return (x, y)

train_rdd = train_df.rdd.map(row_to_xy).cache()
test_rdd = test_df.rdd.map(row_to_xy).cache()

In [13]:
# 12) STANDARDIZE FEATURES USING TRAINING DATA ONLY

def seq_stats(acc, xy):
    count, sum_x, sum_sq = acc
    x, _ = xy
    return (count + 1, sum_x + x, sum_sq + x * x)

def comb_stats(a, b):
    return (a[0] + b[0], a[1] + b[1], a[2] + b[2])

zero = (0, np.zeros(num_features), np.zeros(num_features))
count, sum_x, sum_sq = train_rdd.aggregate(zero, seq_stats, comb_stats)

means = sum_x / count
variances = (sum_sq / count) - (means ** 2)
stds = np.sqrt(np.maximum(variances, 1e-12))
stds = np.where(stds == 0, 1.0, stds)

def standardize_xy(xy):
    x, y = xy
    x_scaled = (x - means) / stds
    return (x_scaled, y)

train_rdd = train_rdd.map(standardize_xy).cache()
test_rdd = test_rdd.map(standardize_xy).cache()

In [14]:
# 12.1) ADD INTERACTION FEATURES

def add_interactions(xy):
    x, y = xy

    interaction1 = x[0] * x[1]   # time_in_hospital * num_medications
    interaction2 = x[2] * x[3]   # inpatient * emergency
    interaction3 = x[4] * x[5]   # outpatient * lab procedures
    interaction4 = x[6] * x[7]   # diagnoses * A1C
    interaction5 = x[8] * x[9]   # glucose * diag_1_group

    x_new = np.append(x, [
        interaction1,
        interaction2,
        interaction3,
        interaction4,
        interaction5
    ])

    return (x_new, y)

train_rdd = train_rdd.map(add_interactions).cache()
test_rdd = test_rdd.map(add_interactions).cache()

num_features = num_features + 5

In [15]:
# 13) ADD INTERCEPT TERM

def add_intercept(xy):
    x, y = xy
    x_new = np.insert(x, 0, 1.0)
    return (x_new, y)

train_rdd = train_rdd.map(add_intercept).cache()
test_rdd = test_rdd.map(add_intercept).cache()

final_num_features = num_features + 1

In [16]:
# 14) LOGISTIC REGRESSION HELPER FUNCTIONS

positive_weight = 8.0
negative_weight = 1.0

def sigmoid(z):
    z = np.clip(z, -30, 30)
    return 1.0 / (1.0 + np.exp(-z))

def compute_gradient_and_loss(record, weights):
    x, y = record
    pred = sigmoid(np.dot(weights, x))

    class_weight = positive_weight if y == 1.0 else negative_weight

    error = class_weight * (pred - y)
    grad = error * x

    eps = 1e-12
    base_loss = -(y * np.log(pred + eps) + (1 - y) * np.log(1 - pred + eps))
    loss = class_weight * base_loss

    return (grad, loss)

def seq_grad_loss(acc, record_weights):
    grad_sum, loss_sum, n = acc
    record, weights = record_weights
    grad, loss = compute_gradient_and_loss(record, weights)
    return (grad_sum + grad, loss_sum + loss, n + 1)

def comb_grad_loss(a, b):
    return (a[0] + b[0], a[1] + b[1], a[2] + b[2])

In [17]:
# 15) TRAIN CUSTOM LOGISTIC REGRESSION

learning_rate = 0.05
epochs = 40
l2_lambda = 0.001

weights = np.zeros(final_num_features)

for epoch in range(1, epochs + 1):
    bc_weights = spark.sparkContext.broadcast(weights)

    zero_acc = (np.zeros(final_num_features), 0.0, 0)

    grad_sum, loss_sum, n = train_rdd.map(
        lambda record: (record, bc_weights.value)
    ).aggregate(
        zero_acc,
        seq_grad_loss,
        comb_grad_loss
    )

    bc_weights.unpersist()

    avg_grad = grad_sum / n
    avg_loss = loss_sum / n

    reg = np.copy(weights)
    reg[0] = 0.0
    avg_grad = avg_grad + l2_lambda * reg

    weights = weights - learning_rate * avg_grad

    print(f"Epoch {epoch:02d} | Training Loss: {avg_loss:.6f}")

print("\nFinal weights:")
print(weights)

Epoch 01 | Training Loss: 1.234620


Epoch 02 | Training Loss: 1.227372


Epoch 03 | Training Loss: 1.223080


Epoch 04 | Training Loss: 1.219697


Epoch 05 | Training Loss: 1.216824


...

Epoch 36 | Training Loss: 1.185269


Epoch 37 | Training Loss: 1.184907


Epoch 38 | Training Loss: 1.184563


Epoch 39 | Training Loss: 1.184235


Epoch 40 | Training Loss: 1.183922

Final weights:
[-0.02643012  0.05700294  0.03967378  0.22747594  0.04371152  0.01517444
  0.01723575  0.06877131 -0.032571    0.0155254  -0.01841906  0.03810317
 -0.00119846 -0.01585756  0.0890836   0.00064295  0.03356939  0.00580068
  0.05088243 -0.01603357  0.00898874  0.01334009  0.00630238 -0.0103656 ]


In [18]:
# 16) PREDICTION FUNCTIONS

def predict_probability(x, weights):
    return float(sigmoid(np.dot(weights, x)))

def predict_label(prob, threshold=0.5):
    return 1 if prob >= threshold else 0


In [19]:
# 17) EVALUATE ON TEST SET

test_predictions = test_rdd.map(
    lambda xy: (
        xy[1],
        predict_probability(xy[0], weights)
    )
).collect()

y_true = [int(t[0]) for t in test_predictions]
y_prob = [float(t[1]) for t in test_predictions]

threshold = 0.5
y_pred = [predict_label(p, threshold) for p in y_prob]

tp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 1 and yp == 1)
tn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 0 and yp == 0)
fp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 0 and yp == 1)
fn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 1 and yp == 0)

accuracy = (tp + tn) / max(len(y_true), 1)
precision = tp / max(tp + fp, 1)
recall = tp / max(tp + fn, 1)
f1 = 2 * precision * recall / max(precision + recall, 1e-12)

def roc_auc_score_manual(y_true, y_scores):
    pairs = list(zip(y_true, y_scores))
    pairs.sort(key=lambda x: x[1])

    n_pos = sum(y_true)
    n_neg = len(y_true) - n_pos

    if n_pos == 0 or n_neg == 0:
        return 0.0

    rank_sum = 0.0
    for i, (label, _) in enumerate(pairs, start=1):
        if label == 1:
            rank_sum += i

    auc = (rank_sum - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
    return auc

auc = roc_auc_score_manual(y_true, y_prob)

print("\n===== TEST RESULTS =====")
print(f"Threshold: {threshold:.2f}")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")

print("\nConfusion Matrix")
print(f"TP: {tp}, TN: {tn}, FP: {fp}, FN: {fn}")


===== TEST RESULTS =====
Threshold: 0.50
Accuracy : 0.6214
Precision: 0.1582
Recall   : 0.5536
F1-score : 0.2461
ROC-AUC  : 0.6302

Confusion Matrix
TP: 1259, TN: 11403, FP: 6698, FN: 1015


In [20]:
# 18) SHOW SAMPLE PREDICTIONS

print("\nSample predictions:")
for i in range(min(10, len(test_predictions))):
    print(f"True={y_true[i]}, Prob={y_prob[i]:.4f}, Pred={y_pred[i]}")


Sample predictions:
True=0, Prob=0.3661, Pred=0
True=0, Prob=0.3475, Pred=0
True=1, Prob=0.3420, Pred=0
True=0, Prob=0.3575, Pred=0
True=0, Prob=0.3486, Pred=0
True=0, Prob=0.4421, Pred=0
True=0, Prob=0.4431, Pred=0
True=0, Prob=0.3551, Pred=0
True=0, Prob=0.3862, Pred=0
True=0, Prob=0.3738, Pred=0


In [21]:
# 19) STOP SPARK

spark.stop()